# 강의용 코랩 | 환경·기후 지표와 기온 예측
교실에서 표를 읽고 그래프를 보며 개념을 설명합니다. 런타임에서 모두 실행을 선택하세요. CSV 자동 로딩에 실패하면 [Drive CSV](https://drive.google.com/uc?export=download&id=1KDArjMKrEFZ4CY2nxvR-H8L6uUDV0Cma)를 세션에 업로드한 뒤 다시 실행합니다.

### 날씨와 기후
대구 관측소에서 어느 날 관측한 기온은 그날의 날씨를 나타냅니다. 해마다 여름이 더운 계절이 반복되는지 살펴보면 계절성을 이해할 수 있습니다. 장기 기후를 설명하려면 수십 년 규모의 기록을 검토합니다. 이번 5년 자료로는 계절 패턴을 살피고 다음 날 기온을 예측하는 원리를 배웁니다.


## 기상자료를 읽는 법

| 지표 | 단위 | 해석 |
|---|---|---|
| 평균기온·최고기온·최저기온 | °C | 평균은 하루의 대표값입니다. 최고·최저는 온도가 어디까지 오르내렸는지 보여줍니다. |
| 일교차 | °C | 같은 날 최고기온 − 최저기온입니다. 하루 평균만으로는 계산할 수 없습니다. |
| 일강수량 | mm | 하루 동안 내린 비 등의 물 깊이입니다. 1 mm는 평평한 1 m²에 물 1 L가 쌓인 깊이입니다. 0과 빈칸의 의미는 자료 설명에서 각각 확인합니다. |
| 상대습도 | % | 같은 온도에서 공기가 포화됐을 때와 비교한 수증기 비율입니다. 100%는 그 온도에서 포화 상태를 뜻합니다. |
| 풍속·풍향 | m/s, 방향 또는 ° | 바람의 세기와 불어오는 방향입니다. 북풍은 북쪽에서 불어옵니다. |
| 기압 | hPa | 공기가 누르는 압력입니다. 현지기압과 해면기압은 기준 높이가 다릅니다. |

실습 CSV에는 **날짜, 시간별 관측에서 집계한 일평균기온, 유효 관측 시간 수**가 들어 있습니다. 최고·최저기온, 강수량, 습도 등은 원자료에서 별도로 확인할 수 있는 지표의 예입니다. 이 CSV의 일평균기온이 공식 일자료의 값과 다를 수 있습니다.

예를 들어 어느 날 최저 4°C, 최고 16°C라면 일교차는 12°C입니다. 시간 관측의 평균이 9°C였다면 하루 평균은 9°C입니다. 관측 시간 수가 23이면 어느 시간대가 빠졌는지에 따라 평균도 달라질 수 있어 실습에서는 그날 값을 결측으로 처리합니다.


## 1. 날짜별 CSV 읽기
2021~2025년 기상청 ASOS 대구(143) 시간자료에서 기온을 날짜별로 집계했습니다. `hours_observed`는 해당 날짜에 실제 기온값이 있는 시간 수입니다.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

filename = "daegu_143_daily_temperature.csv"
paths = [Path(filename), Path("data") / filename]
url = "https://raw.githubusercontent.com/itlects/2026-2_ai-data/main/week11-climate/data/" + filename
source = next((path for path in paths if path.is_file()), url)
raw = pd.read_csv(source)
display(raw.head())
print("관측 날짜 수", len(raw))


## 2. 날짜와 결측 점검
날짜를 정렬하고 달력 기준으로 하루 간격을 맞춥니다. 24시간을 채우지 못한 날은 기온을 결측으로 처리합니다.

In [ ]:
raw["date"] = pd.to_datetime(raw["date"], errors="raise")
assert raw["date"].is_unique, "날짜가 중복되었습니다."
daily = raw.set_index("date").sort_index().asfreq("D")
incomplete = daily["hours_observed"].fillna(0).ne(24)
print("24시간 미만 관측일 수", incomplete.sum())
temp = daily["mean_temp_c"].where(~incomplete)
assert len(temp) == 1826


## 3. 일별 기온과 7일 이동평균 그리기
당일을 포함한 최근 7일을 평균하면 일별 변동이 완화됩니다. 두 곡선의 최고·최저 시점이 어떻게 다른지 확인합니다.

In [ ]:
ax = temp.plot(figsize=(12, 4), alpha=0.45, label="Daily mean")
temp.rolling(7).mean().plot(ax=ax, label="Trailing 7-day mean")
ax.set(ylabel="Temperature (°C)", xlabel="Date")
ax.legend()
plt.show()


### 확인 질문
1. 7일 이동평균이 일별 평균기온보다 완만한 이유를 설명하세요.
2. 24시간 미만 관측일을 왜 0°C로 채우면 안 되는지 적으세요.

## 같은 기온 자료를 세 가지 관점에서 보기
- **변동성**은 하루하루 기온이 오르내리는 정도입니다. 일별 선과 7일 이동평균 선을 비교하세요.
- **계절성**은 해마다 비슷한 시기에 반복되는 패턴입니다. 아래 월별 평균 그래프에서 여름과 겨울을 비교하세요.
- **추세**는 특정 구간에서 값이 한 방향으로 움직이는 경향입니다. 워크북의 선형 추세는 직전 7일만 사용합니다. 같은 5년 자료로 계산한 아래 월별 평균은 장기 기후평년값과 목적과 기간이 다릅니다.


In [ ]:
monthly = temp.groupby(temp.index.month).mean()
ax = monthly.rename_axis(None).plot(use_index=True, x_compat=True, marker="o", xticks=range(1, 13), figsize=(9, 3))
ax.set(xlabel="Month", ylabel="Temperature (°C)", title="Daegu | Mean by calendar month, 2021–2025")
plt.show()


## 두 예측 방법과 오차 지표
- **선형 추세**는 직전 7일 기온에 직선을 맞춰 하루 더 연장합니다.
- **랜덤 포레스트**는 과거 7일 기온과 달력상 계절 위치를 보고 여러 결정 트리의 예측을 평균냅니다.
- **MAE**는 평균적으로 몇 °C 빗나갔는지 보여줍니다. **RMSE**는 큰 오차가 있는 방법에 더 민감합니다.

예를 들어 절대오차가 0°C와 4°C이면 MAE는 2°C, RMSE는 약 2.83°C입니다. 실습의 2025년 대구 365일 평가에서는 선형 추세의 MAE가 2.389°C, 랜덤 포레스트의 MAE가 1.622°C였습니다. 이 차이는 해당 지점과 평가 기간의 결과입니다.

[학생 실습용 코랩 열기](https://colab.research.google.com/github/itlects/2026-2_ai-data/blob/main/week11-climate/02_workbook.ipynb)
